# 🧹 Notebook 02 — Data Preprocessing
Covers: missing values, duplicates, scaling, and train/test split.
---

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

df_raw = pd.read_csv('../data/raw/creditcard.csv')
print(f'Raw shape: {df_raw.shape}')

## 1. Missing Value Check

In [ ]:
missing = df_raw.isnull().sum()
print('Missing values per column:')
print(missing[missing > 0] if missing.sum() > 0 else 'None — dataset is clean! ✓')

## 2. Duplicate Removal

In [ ]:
before = len(df_raw)
df = df_raw.drop_duplicates()
print(f'Rows before: {before:,}  |  After: {len(df):,}  |  Removed: {before-len(df):,}')

## 3. Data Types

In [ ]:
print('Data types:')
print(df.dtypes.value_counts())
print('\nAll numeric?', all(df.dtypes != object))

## 4. Outlier Analysis (IQR Method)

In [ ]:
def iqr_outliers(series):
    Q1, Q3 = series.quantile([0.25, 0.75])
    IQR = Q3 - Q1
    return ((series < Q1 - 1.5*IQR) | (series > Q3 + 1.5*IQR)).sum()

print('Outlier counts (IQR method):')
print(f"  Amount: {iqr_outliers(df['Amount']):,}")
print(f"  Time  : {iqr_outliers(df['Time']):,}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, col in zip(axes, ['Amount','Time']):
    ax.boxplot(df[col], vert=False, patch_artist=True,
               boxprops=dict(facecolor='#3498db', alpha=0.6))
    ax.set_title(f'{col} — Box Plot (with outliers)')
plt.tight_layout()
plt.show()

## 5. Feature Scaling (StandardScaler)

In [ ]:
# V1-V28 are already PCA-normalised. We only scale Time and Amount.
df_scaled = df.copy()
scaler = StandardScaler()
df_scaled[['Time','Amount']] = scaler.fit_transform(df[['Time','Amount']])

print('Before scaling — Amount stats:')
print(df[['Amount']].describe().T)
print('\nAfter scaling — Amount stats:')
print(df_scaled[['Amount']].describe().T)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, data, title in [
    (axes[0], df['Amount'],        'Amount — Before Scaling'),
    (axes[1], df_scaled['Amount'], 'Amount — After Scaling'),
]:
    ax.hist(data, bins=50, color='#3498db', alpha=0.7, edgecolor='white')
    ax.set_title(title)
plt.tight_layout()
plt.show()

## 6. Train / Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X = df_scaled.drop(columns=['Class'])
y = df_scaled['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train: {len(X_train):,} rows  |  Test: {len(X_test):,} rows')
print(f'Train fraud rate: {y_train.mean()*100:.3f}%')
print(f'Test  fraud rate: {y_test.mean()*100:.3f}%')
print('\nStratification preserved fraud ratio ✓')

## 7. SMOTE — Handling Class Imbalance

In [ ]:
from imblearn.over_sampling import SMOTE

print(f'Before SMOTE: Normal={( y_train==0).sum():,} | Fraud={(y_train==1).sum():,}')

smote = SMOTE(random_state=42)
X_res, y_res = smote.fit_resample(X_train, y_train)

print(f'After  SMOTE: Normal={(y_res==0).sum():,} | Fraud={(y_res==1).sum():,}')
print(f'Total training samples: {len(X_res):,}')

# Visualise resampled class balance
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, labels, title in [
    (axes[0], y_train, 'Before SMOTE'),
    (axes[1], y_res,   'After SMOTE'),
]:
    counts = pd.Series(labels).value_counts()
    ax.bar(['Normal','Fraud'], [counts[0], counts[1]],
           color=['#3498db','#e74c3c'], width=0.5)
    ax.set_title(title, fontsize=12)
plt.tight_layout()
plt.show()